# LineFormer GPU inference for GraphDigitization

Runs the **extract** stage of a GraphDigitization run on a Colab GPU (T4 works; its
sm_75 architecture is compatible with the pinned `torch 1.13.1 + cu117` stack that
LineFormer requires).

Workflow:
1. locally: `graphdig run <image> --extractor colab_bundle` (stops after exporting the bundle)
   or `graphdig export-job <run_dir>` for an existing run
2. here: upload `job_bundle_<run_id>.zip`, run all cells, download `results_<run_id>.zip`
3. locally: `graphdig import-results <run_dir> results_<run_id>.zip`
   then `graphdig run --run-dir <run_dir> --stages select,series,qc,report`

The bundle is self-contained: it carries the tiles, the job spec, and the worker script
(`lineformer_infer.py`).

In [ ]:
!nvidia-smi

## 1. Pinned environment (Python 3.10 venv via uv — Colab's system Python is too new)

In [ ]:
%%bash
set -e
curl -LsSf https://astral.sh/uv/install.sh | sh >/dev/null
export PATH="$HOME/.local/bin:$PATH"
uv venv /content/lf --python 3.10 --seed --clear >/dev/null
/content/lf/bin/python -m pip install --quiet torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
    --extra-index-url https://download.pytorch.org/whl/cu117
/content/lf/bin/python -m pip install --quiet -U openmim
/content/lf/bin/python -m mim install "mmcv-full==1.7.1" >/dev/null
/content/lf/bin/python -m pip install --quiet "mmdet==2.28.2" "yapf==0.40.1" gdown opencv-python-headless "numpy<1.24"
/content/lf/bin/python -c "import torch; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())"

## 2. LineFormer code + checkpoint

The checkpoint Google Drive id is in the [LineFormer README](https://github.com/TheJaeLal/LineFormer)
(`iter_3000.pth`). Set `CKPT_GDRIVE_ID` below, or upload the `.pth` into
`/content/LineFormer/` manually.

In [ ]:
CKPT_DRIVE_FOLDER = "https://drive.google.com/drive/folders/1K_zLZwgoUIAJtfjwfCU5Nv33k17R0O5T"  # from the LineFormer README

import glob, os, subprocess
if not os.path.isdir("/content/LineFormer"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/TheJaeLal/LineFormer", "/content/LineFormer"], check=True)
if not glob.glob("/content/LineFormer/**/*.pth", recursive=True):
    subprocess.run(["/content/lf/bin/python", "-m", "gdown", "--folder", CKPT_DRIVE_FOLDER,
                    "-O", "/content/LineFormer/ckpt"], check=True)
print(glob.glob("/content/LineFormer/**/*.pth", recursive=True))

## 3. Upload the job bundle

In [ ]:
from google.colab import files
import zipfile, os, shutil

uploaded = files.upload()  # pick job_bundle_<run_id>.zip
bundle = next(iter(uploaded))
shutil.rmtree("/content/job", ignore_errors=True)
with zipfile.ZipFile(bundle) as zf:
    zf.extractall("/content/job")
import json
job = json.load(open("/content/job/job.json"))
print("run:", job["run_id"], "tiles:", len(job["tiles"]))

## 4. Run inference

In [ ]:
!/content/lf/bin/python /content/job/lineformer_infer.py \
    --job /content/job --out /content/job/results.json \
    --lineformer-dir /content/LineFormer --device cuda:0

## 5. Package + download results

In [ ]:
import zipfile
out = f"/content/results_{job['run_id']}.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("/content/job/results.json", "results.json")
files.download(out)